# colabgame デモ

Google Colab の中で pygame・pyxel・turtle のゲームを動かします。
上から順にセルを実行してください。ゲーム画面は一度クリックするとキー操作が効くようになります。

In [ ]:
!pip install -q git+https://github.com/blackcaty1224/colabgame.git
import colabgame

## turtle：図形を描く
turtle はインストール不要で、すぐに動きます。

In [ ]:
%%game
# アニメーションで図形を描く（turtle版）
import turtle

t = turtle.Turtle()
t.speed(8)
colors = ["red", "orange", "gold", "green", "blue", "purple"]
for i in range(36):
    t.pencolor(colors[i % 6])
    t.circle(80)
    t.left(10)
t.hideturtle()
turtle.done()

## turtle：矢印キーで遊ぶゲーム

In [ ]:
%%game
# 矢印キーでタートルを動かして、赤い点を集めるゲーム（turtle版）
import turtle
import random
import time

wn = turtle.Screen()
wn.title("タートルで点あつめ")
wn.setup(600, 500)
wn.bgcolor("#dff3e4")
wn.tracer(0)

player = turtle.Turtle()
player.shape("turtle")
player.color("#2e7d32")
player.penup()
player.shapesize(1.5)

food = turtle.Turtle()
food.shape("circle")
food.color("crimson")
food.penup()
food.goto(100, 100)

pen = turtle.Turtle()
pen.hideturtle()
pen.penup()
pen.goto(-280, 210)

score = 0
keys = set()

def show_score():
    pen.clear()
    pen.write(f"スコア: {score}", font=("Arial", 18, "bold"))

for k in ("Up", "Down", "Left", "Right"):
    wn.onkeypress(lambda k=k: keys.add(k), k)
    wn.onkeyrelease(lambda k=k: keys.discard(k), k)
wn.listen()
show_score()

while True:
    if "Left" in keys:
        player.left(5)
    if "Right" in keys:
        player.right(5)
    if "Up" in keys:
        player.forward(4)
    if "Down" in keys:
        player.backward(3)
    x, y = player.position()
    if abs(x) > 290 or abs(y) > 240:
        player.goto(max(-290, min(290, x)), max(-240, min(240, y)))
    if player.distance(food) < 20:
        score += 1
        food.goto(random.randint(-260, 260), random.randint(-200, 200))
        show_score()
    wn.update()
    time.sleep(1 / 60)

## pygame
初回は pygbag のインストールとビルドに1分ほどかかります。画面に「Ready to start!」と出たらクリックしてください。

In [ ]:
%%game
# 矢印キーで四角を動かしてコインを集めるゲーム（pygame版）
import pygame
import random

pygame.init()
screen = pygame.display.set_mode((640, 480))
pygame.display.set_caption("コインあつめ")
clock = pygame.time.Clock()
font = pygame.font.Font(None, 40)

player = pygame.Rect(300, 220, 40, 40)
coin = pygame.Rect(random.randint(20, 600), random.randint(20, 440), 20, 20)
score = 0
running = True

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    keys = pygame.key.get_pressed()
    speed = 5
    if keys[pygame.K_LEFT]:
        player.x -= speed
    if keys[pygame.K_RIGHT]:
        player.x += speed
    if keys[pygame.K_UP]:
        player.y -= speed
    if keys[pygame.K_DOWN]:
        player.y += speed
    player.clamp_ip(screen.get_rect())

    if player.colliderect(coin):
        score += 1
        coin.topleft = (random.randint(20, 600), random.randint(20, 440))

    screen.fill((30, 30, 50))
    pygame.draw.rect(screen, (80, 200, 255), player)
    pygame.draw.ellipse(screen, (255, 210, 0), coin)
    screen.blit(font.render(f"SCORE: {score}", True, (255, 255, 255)), (10, 10))
    pygame.display.flip()
    clock.tick(60)

pygame.quit()

## pyxel
初回は pyxel のインストールに少し時間がかかります。

In [ ]:
%%game
# 矢印キーでプレイヤーを動かして、落ちてくる星をキャッチ（pyxel版）
import pyxel
import random


class App:
    def __init__(self):
        pyxel.init(160, 120, title="Star Catch")
        self.x = 72
        self.stars = []
        self.score = 0
        pyxel.run(self.update, self.draw)

    def update(self):
        if pyxel.btn(pyxel.KEY_LEFT):
            self.x = max(self.x - 2, 0)
        if pyxel.btn(pyxel.KEY_RIGHT):
            self.x = min(self.x + 2, pyxel.width - 16)
        if pyxel.frame_count % 20 == 0:
            self.stars.append([random.randint(0, pyxel.width - 4), 0])
        for s in self.stars:
            s[1] += 1.5
        caught = [s for s in self.stars if s[1] > 100 and self.x - 4 < s[0] < self.x + 16]
        self.score += len(caught)
        self.stars = [s for s in self.stars if s not in caught and s[1] < pyxel.height]

    def draw(self):
        pyxel.cls(1)
        for x, y in self.stars:
            pyxel.circ(x + 2, y, 2, 10)
        pyxel.rect(self.x, 104, 16, 6, 11)
        pyxel.text(4, 4, f"SCORE {self.score}", 7)


App()

## ファイルから実行する

`.py` ファイルを Colab にアップロードしてあれば、次のように実行できます。
同じフォルダにある画像や音声のファイルも、コードの中で名前が書かれていれば自動で一緒に読み込まれます。

In [ ]:
# %game_file mygame.py
# %game_file mygame.py --assets images,sounds/bgm.ogg --height 600